# makemore part 2: the MLP, from scratch

This is the *Practice* step of `unit_03_makemore_mlp.md`. Do the Cold Attempt there first.

Work top to bottom. Each milestone is one cell of stubs followed by a grader cell.
The grader stops at your first failure so there is always exactly one thing in front of you.

**Rules of engagement**
- Don't open the lecture. Don't open the makemore repo or the lecture notebook.
- Stuck on an *idea* for 20 min → ask the coaching chat for a hint.
- Stuck on *PyTorch syntax* → ask immediately, zero learning value in that.
- **Before you run a grader cell, say out loud what you expect to happen.**

Everything here is a plain function on plain tensors. The parameters travel around as one dict
(`params`) so each milestone can hand off to the next without a class.

In [ ]:
import random
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

from test_makemore_mlp import grade

## Given: the words and the vocabulary

Plumbing. Read it once so the names below mean something to you, then move on.

In [ ]:
words = open('../data/names.txt').read().splitlines()
print(len(words), words[:8])

chars = sorted(set(''.join(words)))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0                      # '.' is both "start" padding and the "end" token
itos = {i: s for s, i in stoi.items()}
V = len(itos)                      # 27
print(V, itos)

## Milestone 1 — the dataset: a sliding window of context

One name becomes several training examples. Each example is *the last `block_size` characters*
(as integers) and *the character that came next*. The window starts as all padding and the
name has to tell the model when it is over.

In [ ]:
def build_dataset(words, block_size, stoi):
    """Turn a list of names into (X, Y).

    X: int64 tensor (N, block_size). Row i is the window of block_size
       character indices that came right before target i. Before the
       first character of a name the window is all 0 ('.' padding).
    Y: int64 tensor (N,). The index of the character that follows X[i].
    N: sum over words of (len(word) + 1) -- the model must also learn to
       predict the end of a name.
    """
    raise NotImplementedError

In [ ]:
X, Y = build_dataset(words[:3], 3, stoi)
print(X.shape, Y.shape)
for x, y in zip(X[:8], Y[:8]):
    print(''.join(itos[i.item()] for i in x), '-->', itos[y.item()])

In [ ]:
grade(build_dataset, upto=1)

## Milestone 2 — the embedding lookup

27 characters, each gets a small vector. `C` is the table. The question is what `C[X]`
means when `X` is a whole (N, block_size) matrix of indices, and what shape comes out.

In [ ]:
def embed(C, X):
    """Look every index in X up in the table C.

    C: float tensor (V, d)         -- one d-dim vector per character
    X: int64 tensor (N, block_size)
    returns float tensor (N, block_size, d). Must stay differentiable
    w.r.t. C: the table is a parameter and has to learn.
    """
    raise NotImplementedError

In [ ]:
grade(build_dataset, embed, upto=2)

## Milestone 3 — flattening the window (the hard one)

The hidden layer wants one flat vector per example, not `block_size` separate vectors.
`(N, block_size, d)` has to become `(N, block_size * d)`, with example i's embeddings laid
side by side, **in order**, and without copying any memory.

Getting a shape that *looks* right is easy. Getting the numbers in the right places is the
milestone. Try it on a tiny tensor (N=2, block_size=2, d=2) and look at where each element lands.

**flatten_context(emb)**
- Takes: `emb`, float `(N, T, d)`.
- Returns: float `(N, T*d)`. Row i is `[emb[i,0], emb[i,1], ..., emb[i,T-1]]` end to end.
  Must share storage with `emb` (`out.data_ptr() == emb.data_ptr()`): a view, not a copy.
  Gradient flows through it unchanged. `N=1` is an ordinary case, not a special one.
- The grader checks, in order: is a tensor → shape → values (against a slow cat-of-slabs
  reference) → same storage as the input → gradient. A `torch.cat` answer passes the first
  three and is told so; it fails the fourth on purpose.

In [ ]:
def flatten_context(emb):
    """(N, block_size, d) -> (N, block_size * d).

    Row i of the output is [emb[i, 0], emb[i, 1], ..., emb[i, block_size-1]]
    glued end to end. Same storage as emb: no copy.
    """
    raise NotImplementedError

In [ ]:
grade(build_dataset, embed, flatten_context, upto=3)

## Milestone 4 — the forward pass

Given: `init_params`, which builds the parameter dict every later milestone shares. Read the
shapes. Yours: `forward`.

**params** (given, built by `init_params(V, block_size, d, H, g)`)
- `C` `(V, d)`, `W1` `(block_size*d, H)`, `b1` `(H,)`, `W2` `(H, V)`, `b2` `(V,)`.
  All `requires_grad=True`. Nothing else is in the dict.

**forward(X, params)**
- Takes: `X`, int64 `(N, block_size)`; `params`, the dict above.
- Computes: `tanh(flatten_context(embed(C, X)) @ W1 + b1) @ W2 + b2`.
- Returns: float `(N, V)`, raw logits (not squashed, not normalised), still attached to
  every parameter in the graph. Works for any `block_size`, `d`, `H` the dict implies; the
  function must not hard-code any of them.
- The grader checks, in order: is a tensor → shape `(N, V)` → not squashed into (-1, 1) →
  not the tanh-less linear network → values match the reference to 1e-4 → `requires_grad`.

In [ ]:
def init_params(V, block_size, d, H, g):
    """Given. Random-normal init, everything requires_grad.

    C  (V, d)               the embedding table
    W1 (block_size * d, H)  hidden layer weights     b1 (H,)
    W2 (H, V)               output layer weights     b2 (V,)
    """
    params = {
        'C':  torch.randn((V, d), generator=g),
        'W1': torch.randn((block_size * d, H), generator=g),
        'b1': torch.randn(H, generator=g),
        'W2': torch.randn((H, V), generator=g),
        'b2': torch.randn(V, generator=g),
    }
    for p in params.values():
        p.requires_grad = True
    return params


def forward(X, params):
    """X: int64 (N, block_size)  ->  logits: float (N, V).

    embed -> flatten -> hidden layer with a tanh -> output layer.
    The output is raw scores (logits), not probabilities.
    """
    raise NotImplementedError

In [ ]:
g = torch.Generator().manual_seed(2147483647)
params = init_params(V, block_size=3, d=2, H=100, g=g)
print(sum(p.numel() for p in params.values()), 'parameters')
grade(build_dataset, embed, flatten_context, forward, upto=4)

## Milestone 5 — cross-entropy, by hand

Logits to probabilities to "how surprised was the model by the right answer", averaged over the
batch. Write it out of `exp`, sums, indexing and `log`; the grader compares it to
`F.cross_entropy` on the same inputs, including on inputs that are numerically nasty.

**cross_entropy(logits, Y)**
- Takes: `logits`, float `(N, V)`; `Y`, int64 `(N,)`, one target index per row.
- Computes: the mean over rows of `-log(softmax(logits)[i, Y[i]])`.
- Returns: a 0-d float tensor, attached to `logits` in the graph, equal to
  `F.cross_entropy(logits, Y)` to 1e-4. Must be finite and still equal when every logit is
  offset by +100. `N=1` is an ordinary case. No `F.cross_entropy`, `log_softmax`, or
  `softmax` calls.
- The grader checks, in order: is a tensor → 0-d → not the batch total → not sign-flipped →
  equals torch on four batch sizes → gradient w.r.t. logits equals torch's → finite on +100
  logits → equals torch on +100 logits.

In [ ]:
def cross_entropy(logits, Y):
    """Mean negative log-likelihood of the targets.

    logits: float (N, V)   Y: int64 (N,)
    returns a 0-d tensor, still attached to the graph.
    Must give the same number as F.cross_entropy(logits, Y), also when
    the logits are large.
    """
    raise NotImplementedError

In [ ]:
grade(build_dataset, embed, flatten_context, forward, cross_entropy, upto=5)

## Milestone 6 — minibatch training, and it learns

Given: the 80/10/10 split and `split_loss`. Yours: the loop. The grader trains a small model
with your loop for 4000 steps of batch 32 at lr 0.1 and checks the dev loss afterwards.

**train(params, X, Y, steps, batch_size, lr, g)**
- Takes: `params`, the milestone-4 dict; `X` int64 `(N, block_size)`, `Y` int64 `(N,)`, the
  full training split; `steps`, `batch_size` ints; `lr` float; `g`, a `torch.Generator` that
  every random draw must use.
- Each step: `batch_size` random row indices from `[0, N)` drawn with `g` (with
  replacement), loss = `cross_entropy(forward(X[ix], params), Y[ix])`, then every tensor in
  `params` moves by `-lr * grad` in place.
- Returns: a list of exactly `steps` Python floats, the minibatch loss at each step in
  order. After it returns, every tensor in `params` is still the same object, still
  `requires_grad=True`, and has moved.
- The grader checks, in order: returns a list of `steps` items → all Python floats → no
  NaN / blow-up → every one of the five parameters changed → all still `requires_grad` →
  your `forward`+`cross_entropy` agree with the reference on the dev set → dev loss under
  2.95 (a correct loop lands near 2.6; a fresh init is 15–20).

In [ ]:
def split_words(words, seed=42):
    """Given. Shuffle, then 80% train / 10% dev / 10% test."""
    ws = list(words)
    random.Random(seed).shuffle(ws)
    n1, n2 = int(0.8 * len(ws)), int(0.9 * len(ws))
    return ws[:n1], ws[n1:n2], ws[n2:]


@torch.no_grad()
def split_loss(X, Y, params):
    """Given. Loss on a whole split, no graph."""
    return cross_entropy(forward(X, params), Y).item()


def train(params, X, Y, steps, batch_size, lr, g):
    """Plain SGD on random minibatches.

    Each step: draw batch_size random row indices (use the generator g),
    forward + loss on that batch, backward, update every tensor in params
    in place from its own .grad.
    Returns a list of `steps` Python floats: the minibatch loss at each step.
    """
    raise NotImplementedError


block_size = 3
train_words, dev_words, test_words = split_words(words)
Xtr, Ytr = build_dataset(train_words, block_size, stoi)
Xdev, Ydev = build_dataset(dev_words, block_size, stoi)
Xte, Yte = build_dataset(test_words, block_size, stoi)
print(Xtr.shape, Xdev.shape, Xte.shape)

In [ ]:
grade(build_dataset, embed, flatten_context, forward, cross_entropy, train, upto=6)

## Milestone 7 (stretch) — sampling

Turn the model around: start from an all-padding window and let it write a name one character
at a time. The window has to move the same way it did in `build_dataset`.

**sample(params, itos, block_size, g, max_len=30)**
- Takes: `params`, the milestone-4 dict; `itos`, index → char; `block_size` int; `g`, a
  `torch.Generator` that every draw must use; `max_len`, the most characters to emit.
- Each step: `forward` on the current window as a `(1, block_size)` int64 tensor, softmax
  the one row of logits, draw one index with `torch.multinomial(..., generator=g)`, then
  the window slides so the drawn index is in the last slot.
- Returns: a `str` of the drawn characters in order, WITHOUT the '.' that ended it. Stops at
  the first draw of index 0 or after `max_len` characters, whichever comes first; an
  immediate '.' returns `""`.
- The grader checks, in order (on a rigged model whose only rule is "next = previous + 1"):
  is a str → no '.' inside → the window actually slides → the string is exactly the
  alphabet. Then on the model milestone 6 trained: only a–z → 20 draws give ≥10 distinct
  names (drawn, not argmax) → average length 2–12.

In [ ]:
def sample(params, itos, block_size, g, max_len=30):
    """Generate one name. Returns it as a str WITHOUT the trailing '.'.

    Start from a context of block_size zeros. Repeat: forward the current
    window, turn logits into probabilities, draw one index with
    torch.multinomial(..., generator=g). Stop on '.' (index 0) or after
    max_len characters.
    """
    raise NotImplementedError

The final grade cell runs everything and skips milestone 7, so the unit can be
"all milestones passed" without the stretch. Nothing later in the grader needs
`sample`; the only later user of it is part 3 of the Output section (ten names),
and milestone 7 needs the params that milestone 6 leaves behind, so 6 must run in
the same call. When you attempt the stretch, change `skip=(7,)` to `skip=()`.

In [ ]:
grade(build_dataset, embed, flatten_context, forward, cross_entropy, train, sample, skip=(7,))

## Output — pick a learning rate, train for real, look at what it learned

From memory, no grader. Three parts:

1. **Learning-rate sweep.** Train fresh models for ~1000 steps each at learning rates spaced
   *exponentially* between 0.001 and 1 (the `lrs` line is given). Plot loss against the exponent
   and choose an lr from the plot. Decide for yourself what "good" looks like on that curve.
2. **Train longer** with the lr you chose (tens of thousands of steps is still seconds), then
   drop the lr for the last chunk. Report train / dev / test loss. Say *before* you look which
   of the three you expect to be lowest and why.
3. **Look at the embedding.** `plot_embedding` is given; it only works for `d=2`. Which
   characters landed near each other? Sample ten names.

In [ ]:
def plot_embedding(C, itos):
    """Given. Scatter the 2-d embedding with each character labelled."""
    C = C.detach()
    assert C.shape[1] == 2, "plot_embedding only knows how to draw d=2"
    plt.figure(figsize=(6, 6))
    plt.scatter(C[:, 0], C[:, 1], s=200)
    for i in range(C.shape[0]):
        plt.text(C[i, 0].item(), C[i, 1].item(), itos[i], ha='center', va='center', color='white')
    plt.grid('minor')
    plt.show()


# 1. sweep
lre = torch.linspace(-3, 0, 1000)   # exponents
lrs = 10 ** lre                     # learning rates, exponentially spaced
...

In [ ]:
# 2. train longer with the lr you chose, then decay it
...

In [ ]:
# 3. embedding plot and ten samples
...

## Scratch

Space to poke at things. `emb.storage()` / `emb.stride()` / `emb.is_contiguous()` are worth a look
if milestone 3 surprised you.